# STEP 2. 결합 — 요식업 패널 만들기

## 결합 설계

| 대상 | 결합 키 |
|---|---|
| 영역-상권 | `상권_코드` |
| 길단위인구 · 집객시설 | `상권_코드` + `기준_년분기_코드` |
| 추정매출 | `상권_코드` + `기준_년분기_코드` + `서비스_업종_코드` |

## 검증 원칙
좌결합(left join) 후 **행 수가 늘면 키 중복** → 즉시 중단합니다. 이 단계의 최대 위험입니다.

In [2]:
import sys, os
from pathlib import Path

# 노트북이 어디서 열리든 프로젝트 루트를 찾아 sys.path에 추가
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
print("프로젝트 루트:", ROOT)

프로젝트 루트: c:\Users\spide\ai-data-bootcamp\project\h2_nb


In [3]:
from config import PROC, FOOD

store    = pd.read_pickle(PROC / "01_store.pkl")
sales    = pd.read_pickle(PROC / "01_sales.pkl")
area     = pd.read_pickle(PROC / "01_area.pkl")
flow     = pd.read_pickle(PROC / "01_flow.pkl")
facility = pd.read_pickle(PROC / "01_facility.pkl")

def check_rows(before, after, step):
    if after != before:
        raise AssertionError(f"[{step}] 행 증식: {before:,} → {after:,}. 키 중복 확인 필요")
    print(f"  [{step:12}] 행 유지 {after:,}  OK")

print("요식업 10종:", FOOD)

요식업 10종: ['한식음식점', '중식음식점', '일식음식점', '양식음식점', '분식전문점', '패스트푸드점', '치킨전문점', '제과점', '커피-음료', '호프-간이주점']


## 2-1. 요식업 필터

In [5]:
n_all = len(store)
df = store[store["서비스_업종_코드_명"].isin(FOOD)].copy()
print(f"전체 업종 {n_all:,}행 → 요식업 {len(df):,}행 ({len(df)/n_all:.1%})\n")
print(df["서비스_업종_코드_명"].value_counts().to_string())

전체 업종 995,377행 → 요식업 160,352행 (16.1%)

서비스_업종_코드_명
한식음식점      20474
커피-음료      19757
분식전문점      17509
호프-간이주점    17171
제과점        15538
치킨전문점      15035
중식음식점      14323
패스트푸드점     14164
양식음식점      13494
일식음식점      12887


## 2-2. 상권 단위 집계 (밀도 분모용)

`상권_전체점포` 는 요식업 필터 **전**의 100개 업종 합으로 계산해야 정확합니다.

In [6]:
tot_all = (store.groupby(["기준_년분기_코드", "상권_코드"])["점포_수"]
           .sum().rename("상권_전체점포").reset_index())
tot_food = (store[store["서비스_업종_코드_명"].isin(FOOD)]
            .groupby(["기준_년분기_코드", "상권_코드"])["점포_수"]
            .sum().rename("상권_외식점포").reset_index())
print(f"상권_전체점포 (100개 업종 합) : {len(tot_all):,}개 상권×분기")
print(f"상권_외식점포 (요식 10종 합)  : {len(tot_food):,}개 상권×분기")

상권_전체점포 (100개 업종 합) : 21,450개 상권×분기
상권_외식점포 (요식 10종 합)  : 21,212개 상권×분기


## 2-3. 좌결합

In [7]:
n = len(df)

df = df.merge(
    area[["상권_코드", "상권_구분_코드_명", "자치구_코드_명", "행정동_코드_명",
          "영역_면적", "엑스좌표_값", "와이좌표_값"]]
    .rename(columns={"상권_구분_코드_명": "상권유형", "자치구_코드_명": "자치구"}),
    on="상권_코드", how="left")
check_rows(n, len(df), "영역")

df = df.merge(flow[["기준_년분기_코드", "상권_코드", "총_유동인구_수"]],
              on=["기준_년분기_코드", "상권_코드"], how="left")
check_rows(n, len(df), "길단위인구")

df = df.merge(facility[["기준_년분기_코드", "상권_코드", "집객시설_수", "지하철_역_수"]],
              on=["기준_년분기_코드", "상권_코드"], how="left")
check_rows(n, len(df), "집객시설")

df = df.merge(sales[["기준_년분기_코드", "상권_코드", "서비스_업종_코드",
                     "당월_매출_금액", "당월_매출_건수"]],
              on=["기준_년분기_코드", "상권_코드", "서비스_업종_코드"], how="left")
check_rows(n, len(df), "추정매출")

df = df.merge(tot_all,  on=["기준_년분기_코드", "상권_코드"], how="left")
df = df.merge(tot_food, on=["기준_년분기_코드", "상권_코드"], how="left")
check_rows(n, len(df), "상권집계")

  [영역          ] 행 유지 160,352  OK
  [길단위인구       ] 행 유지 160,352  OK
  [집객시설        ] 행 유지 160,352  OK
  [추정매출        ] 행 유지 160,352  OK
  [상권집계        ] 행 유지 160,352  OK


## 2-4. 결측률 — H2 매칭 가능 여부 판단

결측률이 높은 변수는 매칭 공변량에서 빼야 합니다. 표본이 통째로 날아가기 때문입니다.

In [9]:
rows = []
for c in ["엑스좌표_값", "영역_면적", "총_유동인구_수", "집객시설_수",
          "지하철_역_수", "당월_매출_금액", "상권_전체점포"]:
    r = df[c].isna().mean()
    판정 = "OK" if r < 0.05 else ("주의" if r < 0.3 else "매칭 제외 권장")
    rows.append({"컬럼": c, "결측률": r, "판정": 판정})

result = pd.DataFrame(rows)
result["결측률"] = result["결측률"].apply(lambda x: f"{x:.1%}")
display(result)

,컬럼,결측률,판정
0,엑스좌표_값,0.0%,OK
1,영역_면적,0.0%,OK
2,총_유동인구_수,0.0%,OK
3,집객시설_수,2.3%,OK
4,지하철_역_수,85.0%,매칭 제외 권장
5,당월_매출_금액,45.4%,매칭 제외 권장
6,상권_전체점포,0.0%,OK


## 2-5. 분기 커버리지

In [10]:
cov = df.groupby("기준_년분기_코드").agg(
    행수=("점포_수", "size"),
    유동인구결측=("총_유동인구_수", lambda s: s.isna().mean()),
    매출결측=("당월_매출_금액", lambda s: s.isna().mean()))
display(cov.round(3))

df.to_pickle(PROC / "02_merged.pkl")
print(f"[저장] 02_merged.pkl  {df.shape}")

,행수,유동인구결측,매출결측
기준_년분기_코드,,,
20231,12397,0.000,0.446
20232,12409,0.000,0.448
20233,12393,0.000,0.447
20234,12405,0.000,0.448
20241,12418,0.000,0.450
20242,12388,0.000,0.452
20243,12348,0.001,0.453
20244,12320,0.000,0.455
20251,12304,0.000,0.459


[저장] 02_merged.pkl  (160352, 26)
